# 66 - Contrarian Signal Analysis: Buy When Signal Says Sell?

**Surprising Finding from notebook 65:**

| Decile | Signal Range | 90d Forward Return |
|--------|--------------|--------------------|
| 0 (Most Bullish) | -2.16 to -1.44 | +30.0% |
| 9 (Most Bearish) | 0.99 to 1.76 | **+45.5%** |

**The 'overheated' signal has the BEST forward returns!**

Possible explanations:
1. **Momentum**: Bitcoin runs higher than fundamentals suggest
2. **Lagging**: By the time signal is 'hot', the move is just starting
3. **Mean reversion timing**: Corrections are short, then rally resumes

Let's test a **contrarian** approach: Buy when signal is high (bearish).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite signal (same as before)
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

for metric in CONFIG.keys():
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['signal'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

## Part 1: Deep Dive into the Surprising Result

In [ ]:
# Forward returns at multiple horizons
for days in [7, 14, 30, 60, 90, 180, 365]:
    df[f'fwd_{days}d'] = df['price'].shift(-days) / df['price'] - 1

# Create deciles
df['signal_decile'] = pd.qcut(df['signal'], 10, labels=False, duplicates='drop')

print("FORWARD RETURNS BY SIGNAL DECILE - ALL HORIZONS")
print("="*100)
print(f"{'Decile':<8} {'Signal':<18} {'7d':>10} {'14d':>10} {'30d':>10} {'90d':>10} {'180d':>10} {'365d':>10}")
print("-"*100)

for decile in range(10):
    mask = df['signal_decile'] == decile
    sig_min = df.loc[mask, 'signal'].min()
    sig_max = df.loc[mask, 'signal'].max()
    
    row = f"{decile:<8} [{sig_min:>5.2f}, {sig_max:>5.2f}]   "
    for days in [7, 14, 30, 90, 180, 365]:
        ret = df.loc[mask, f'fwd_{days}d'].mean() * 100
        row += f"{ret:>+9.1f}%"
    print(row)

print("-"*100)

In [ ]:
# Visualize the counterintuitive result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 30-day forward returns by decile
decile_returns_30 = df.groupby('signal_decile')['fwd_30d'].mean() * 100
colors = ['#22c55e' if i < 3 else '#ef4444' if i > 6 else '#6b7280' for i in range(10)]
axes[0].bar(range(10), decile_returns_30.values, color=colors, edgecolor='white')
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(['Bullish\n(Buy?)'] + [str(i) for i in range(1, 9)] + ['Bearish\n(Sell?)'])
axes[0].set_ylabel('30-Day Forward Return (%)')
axes[0].set_title('30-Day Returns: No Clear Pattern', fontsize=12, fontweight='bold')
axes[0].axhline(y=0, color='white', linestyle='--', alpha=0.3)

# 90-day forward returns by decile
decile_returns_90 = df.groupby('signal_decile')['fwd_90d'].mean() * 100
axes[1].bar(range(10), decile_returns_90.values, color=colors, edgecolor='white')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(['Bullish\n(Buy?)'] + [str(i) for i in range(1, 9)] + ['Bearish\n(Sell?)'])
axes[1].set_ylabel('90-Day Forward Return (%)')
axes[1].set_title('90-Day Returns: BEARISH SIGNAL = BEST RETURNS!', fontsize=12, fontweight='bold', color='#ef4444')
axes[1].axhline(y=0, color='white', linestyle='--', alpha=0.3)

for i, v in enumerate(decile_returns_90.values):
    axes[1].text(i, v + 1, f'{v:.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Part 2: When Does the High Signal Occur?

In [ ]:
# Find periods when signal was > 1.0 ("bearish")
high_signal = df[df['signal'] > 1.0].copy()

print("PERIODS WITH HIGH SIGNAL (>1.0)")
print("="*80)
print(f"Total days: {len(high_signal)} ({len(high_signal)/len(df)*100:.1f}% of all time)")
print(f"\nDate ranges:")

# Find continuous periods
high_signal['gap'] = (high_signal.index.to_series().diff() > pd.Timedelta(days=7)).cumsum()
periods = high_signal.groupby('gap').agg(
    start=('price', lambda x: x.index[0]),
    end=('price', lambda x: x.index[-1]),
    days=('price', 'count'),
    avg_signal=('signal', 'mean'),
    start_price=('price', 'first'),
    end_price=('price', 'last')
)

print(f"\n{'Start':<12} {'End':<12} {'Days':>6} {'Signal':>8} {'Start $':>10} {'End $':>10} {'Return':>10}")
print("-"*80)
for _, p in periods.iterrows():
    ret = (p['end_price'] / p['start_price'] - 1) * 100
    print(f"{p['start'].strftime('%Y-%m-%d'):<12} {p['end'].strftime('%Y-%m-%d'):<12} {p['days']:>6} "
          f"{p['avg_signal']:>+7.2f} {p['start_price']:>10,.0f} {p['end_price']:>10,.0f} {ret:>+9.1f}%")

In [ ]:
# What happens AFTER high signal periods?
print("\nWHAT HAPPENS AFTER HIGH SIGNAL PERIODS?")
print("="*80)

# Get end dates of high signal periods
period_ends = periods['end'].values

results = []
for end_date in period_ends:
    if end_date in df.index:
        idx = df.index.get_loc(end_date)
        if idx + 90 < len(df):
            fwd_30 = (df.iloc[idx + 30]['price'] / df.iloc[idx]['price'] - 1) * 100
            fwd_90 = (df.iloc[idx + 90]['price'] / df.iloc[idx]['price'] - 1) * 100
            results.append({'date': end_date, 'fwd_30': fwd_30, 'fwd_90': fwd_90})

if results:
    results_df = pd.DataFrame(results)
    print(f"\nAfter {len(results)} high-signal periods ended:")
    print(f"  Average 30-day forward return: {results_df['fwd_30'].mean():+.1f}%")
    print(f"  Average 90-day forward return: {results_df['fwd_90'].mean():+.1f}%")
    print(f"  Win rate (90d > 0): {(results_df['fwd_90'] > 0).mean()*100:.0f}%")

## Part 3: Test Different Signal Thresholds

In [ ]:
# Test various BUY thresholds (higher = more contrarian)
thresholds = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

print("\nFORWARD RETURNS BY BUY THRESHOLD")
print("="*80)
print(f"{'Buy When Signal <':<20} {'Days':>8} {'30d Fwd':>12} {'90d Fwd':>12} {'180d Fwd':>12}")
print("-"*80)

for thresh in thresholds:
    mask = df['signal'] < thresh
    days = mask.sum()
    if days > 0:
        fwd_30 = df.loc[mask, 'fwd_30d'].mean() * 100
        fwd_90 = df.loc[mask, 'fwd_90d'].mean() * 100
        fwd_180 = df.loc[mask, 'fwd_180d'].mean() * 100
        print(f"  Signal < {thresh:>5.1f}        {days:>8} {fwd_30:>+11.1f}% {fwd_90:>+11.1f}% {fwd_180:>+11.1f}%")

print("\n" + "-"*80)
print(f"{'Buy When Signal >':<20} {'Days':>8} {'30d Fwd':>12} {'90d Fwd':>12} {'180d Fwd':>12}")
print("-"*80)

for thresh in thresholds:
    mask = df['signal'] > thresh
    days = mask.sum()
    if days > 0:
        fwd_30 = df.loc[mask, 'fwd_30d'].mean() * 100
        fwd_90 = df.loc[mask, 'fwd_90d'].mean() * 100
        fwd_180 = df.loc[mask, 'fwd_180d'].mean() * 100
        print(f"  Signal > {thresh:>5.1f}        {days:>8} {fwd_30:>+11.1f}% {fwd_90:>+11.1f}% {fwd_180:>+11.1f}%")

## Part 4: Backtest Contrarian Strategy

In [ ]:
def backtest_signal_strategy(df, buy_threshold, sell_threshold, name, min_position=0.0):
    """
    Backtest a simple signal-based strategy.
    buy_threshold: Buy when signal CROSSES BELOW this
    sell_threshold: Sell when signal CROSSES ABOVE this
    """
    bt = df[['price', 'returns', 'signal']].copy()
    
    # Position: 1 when signal < buy_threshold, 0 when signal > sell_threshold
    bt['raw_position'] = 0.0
    bt.loc[bt['signal'] < buy_threshold, 'raw_position'] = 1.0
    bt.loc[bt['signal'] > sell_threshold, 'raw_position'] = min_position
    
    # Forward fill position (stay in trade until exit signal)
    bt['position'] = bt['raw_position'].replace(0, np.nan).ffill().fillna(min_position)
    bt['position'] = bt['position'].shift(1)  # Use previous day's signal
    
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    
    return {
        'name': name,
        'buy_thresh': buy_threshold,
        'sell_thresh': sell_threshold,
        'total_return': (bt['equity'].iloc[-1] / 100000 - 1) * 100,
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'avg_position': bt['position'].mean(),
        'equity': bt['equity'],
        'dd': bt['dd'],
        'position': bt['position']
    }

# HODL baseline
hodl = 100000 * (1 + df['returns']).cumprod()
hodl_dd = hodl / hodl.cummax() - 1
years = (df.index[-1] - df.index[0]).days / 365

# Test various strategies
strategies = [
    # Traditional: Buy low signal, sell high signal
    (-1.0, 1.0, 'Traditional [-1, +1]'),
    (-0.5, 0.5, 'Traditional [-0.5, +0.5]'),
    
    # Contrarian: Buy HIGH signal, sell LOW signal
    (1.0, -0.5, 'Contrarian [+1, -0.5]'),
    (0.5, -1.0, 'Contrarian [+0.5, -1]'),
    (0.75, 0.0, 'Contrarian [+0.75, 0]'),
    
    # Momentum: Stay in when signal rising
    (0.0, 1.5, 'Momentum [0, +1.5]'),
]

results = []
for buy_t, sell_t, name in strategies:
    result = backtest_signal_strategy(df, buy_t, sell_t, name)
    results.append(result)

In [ ]:
print("\n" + "="*100)
print("STRATEGY COMPARISON: TRADITIONAL vs CONTRARIAN")
print("="*100)

hodl_cagr = ((hodl.iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_sharpe = (df['returns'].mean() / df['returns'].std()) * np.sqrt(365)
hodl_max_dd = hodl_dd.min() * 100

print(f"\n{'Strategy':<25} {'Buy<':>8} {'Sell>':>8} {'CAGR':>10} {'MaxDD':>10} {'Sharpe':>10} {'AvgPos':>10}")
print("-"*100)
print(f"{'HODL':<25} {'--':>8} {'--':>8} {hodl_cagr:>9.1f}% {hodl_max_dd:>9.1f}% {hodl_sharpe:>10.2f} {'100%':>10}")
print("-"*100)

for r in results:
    print(f"{r['name']:<25} {r['buy_thresh']:>8.1f} {r['sell_thresh']:>8.1f} "
          f"{r['cagr']:>9.1f}% {r['max_dd']:>9.1f}% {r['sharpe']:>10.2f} {r['avg_position']:>9.0%}")

print("-"*100)
best = max(results, key=lambda x: x['sharpe'])
print(f"\n🏆 Best Risk-Adjusted: {best['name']} (Sharpe: {best['sharpe']:.2f})")

In [ ]:
# Visualize best strategies
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Equity curves
axes[0].semilogy(hodl.index, hodl, 'orange', linewidth=2, label=f'HODL ({hodl_cagr:.0f}%)')
colors = ['#22c55e', '#3b82f6', '#ef4444', '#a855f7', '#f59e0b', '#ec4899']
for r, c in zip(results, colors):
    axes[0].semilogy(r['equity'].index, r['equity'], color=c, linewidth=1.5, 
                     label=f"{r['name']} ({r['cagr']:.0f}%)")

axes[0].set_ylabel('Equity ($)')
axes[0].set_title('Traditional vs Contrarian Strategies', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.3)

# Signal with buy/sell zones
axes[1].plot(df.index, df['signal'], color='white', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='white', linestyle='-', alpha=0.3)
axes[1].axhline(y=1.0, color='#ef4444', linestyle='--', linewidth=2, label='Contrarian BUY (>1.0)')
axes[1].axhline(y=-1.0, color='#22c55e', linestyle='--', linewidth=2, label='Traditional BUY (<-1.0)')
axes[1].fill_between(df.index, 1.0, df['signal'].max(), alpha=0.2, color='#ef4444')
axes[1].fill_between(df.index, -1.0, df['signal'].min(), alpha=0.2, color='#22c55e')
axes[1].set_ylabel('Signal')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Understanding WHY Contrarian Works

In [ ]:
# Analyze what happens at different signal levels
print("\nANALYZING SIGNAL BEHAVIOR")
print("="*80)

# Calculate signal momentum
df['signal_change'] = df['signal'].diff()
df['signal_momentum'] = df['signal'].diff(7)  # 7-day change

# When is signal > 1.0?
high_signal = df['signal'] > 1.0

print(f"\nWhen signal > 1.0 ('overheated'):")
print(f"  Avg price level: ${df.loc[high_signal, 'price'].mean():,.0f}")
print(f"  Avg MVRV: {df.loc[high_signal, 'mvrv'].mean():.2f}")
print(f"  Signal often FALLING: {(df.loc[high_signal, 'signal_change'] < 0).mean()*100:.0f}% of the time")

# What's the signal doing when returns are best?
top_returns = df['fwd_90d'] > df['fwd_90d'].quantile(0.9)
print(f"\nWhen 90d forward returns are in top 10%:")
print(f"  Average signal level: {df.loc[top_returns, 'signal'].mean():.2f}")
print(f"  Signal distribution:")
print(f"    < -1.0 (bullish): {(df.loc[top_returns, 'signal'] < -1.0).mean()*100:.1f}%")
print(f"    -1.0 to 0: {((df.loc[top_returns, 'signal'] >= -1.0) & (df.loc[top_returns, 'signal'] < 0)).mean()*100:.1f}%")
print(f"    0 to 1.0: {((df.loc[top_returns, 'signal'] >= 0) & (df.loc[top_returns, 'signal'] < 1.0)).mean()*100:.1f}%")
print(f"    > 1.0 (bearish): {(df.loc[top_returns, 'signal'] > 1.0).mean()*100:.1f}%")

In [ ]:
# The key insight: Look at SIGNAL DIRECTION not just level
print("\nSIGNAL DIRECTION ANALYSIS")
print("="*80)

# Create signal direction indicator
df['signal_rising'] = df['signal_momentum'] > 0

# Compare returns based on level AND direction
conditions = [
    ('Low signal, rising', (df['signal'] < 0) & df['signal_rising']),
    ('Low signal, falling', (df['signal'] < 0) & ~df['signal_rising']),
    ('High signal, rising', (df['signal'] > 0.5) & df['signal_rising']),
    ('High signal, falling', (df['signal'] > 0.5) & ~df['signal_rising']),
]

print(f"\n{'Condition':<25} {'Days':>8} {'30d Fwd':>12} {'90d Fwd':>12}")
print("-"*60)
for name, mask in conditions:
    days = mask.sum()
    fwd_30 = df.loc[mask, 'fwd_30d'].mean() * 100
    fwd_90 = df.loc[mask, 'fwd_90d'].mean() * 100
    print(f"{name:<25} {days:>8} {fwd_30:>+11.1f}% {fwd_90:>+11.1f}%")

## Summary & Conclusions

In [ ]:
print("\n" + "#"*80)
print("KEY FINDINGS")
print("#"*80)

print("""
SURPRISING RESULT:
  The 'bearish' signal (>1.0) actually precedes the BEST forward returns!
  
POSSIBLE EXPLANATIONS:

1. MOMENTUM DOMINATES
   - Bitcoin trends strongly
   - "Overvalued" can stay overvalued for months
   - The signal catches the MIDDLE of moves, not the END

2. SIGNAL IS LAGGING
   - On-chain data reflects past activity
   - By the time signal says "hot", smart money already positioned
   - Price continues higher driven by momentum

3. SAMPLE BIAS
   - High signal periods often occur in bull markets
   - Bull markets have positive forward returns regardless
   - Need to control for market regime

IMPLICATIONS FOR TRADING:

  ❌ DON'T: Sell when signal says "overheated"
  ✅ DO: Use signal for RISK MANAGEMENT, not market timing
  ✅ DO: Consider contrarian positions when signal extreme
  ✅ DO: Look at signal DIRECTION not just level
""")

# Current state
latest = df.iloc[-1]
print(f"\nCURRENT STATE ({latest.name.date()}):")
print(f"  Signal: {latest['signal']:+.2f}")
print(f"  Signal momentum (7d): {latest['signal_momentum']:+.3f}")

if latest['signal'] > 1.0:
    print(f"\n  ⚠️ Signal is 'bearish' but historically this precedes STRONG returns!")
elif latest['signal'] < -1.0:
    print(f"\n  ✅ Signal is 'bullish' - traditional accumulation zone")
else:
    print(f"\n  ⚪ Signal is neutral")